## HIL-005 Data Cleaning

This notebook documents the cleaning and validation process for dataset HIL-005.  The raw source data will remain unchanged, and all cleaned outputs will be saved separately.

Source: City of Hillsboro GIS

Dataset: Buildings (<1,000)

Layer ID: 91

Geometry: Polygon

Spatial Reference: WKID 3857

Source documentation: [\[gis.hillsboro-oregon.gov\]](https://gis.hillsboro-oregon.gov/public/rest/services/public/Planning_BaseData/MapServer/91)

## Findings from the HIL_005_Buildings_Exploration Notebook

- The layer contains 43,686 building records.
- Geometry type is polygon.
- The current dataset contains no Demoed or Permitted records.
- `YEAR_BUILT = 0` occurs in 8,732 records (~20% of the dataset) and should be treated as a potential missing/unknown value rather than a literal construction year.
- `YEAR_DEMOLISHED` is not populated in the current dataset.
- The `STATUS` field uses a coded domain: 0 = Active, 1 = Demoed, 2 = Permitted.
- The current dataset therefore appears most useful for analyzing the existing building stock rather than historical demolition activity.

In [ ]:
from pathlib import Path

# Establish the project root
PROJECT_ROOT = Path.cwd().parent

# Locate available dated data folders
DATA_FOLDERS = sorted(
    [
        folder
        for folder in PROJECT_ROOT.iterdir()
        if folder.is_dir() and folder.name.startswith("20")
    ]
)

print("Available data folders:")
for folder in DATA_FOLDERS:
    print("-", folder.name)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Change to the desired data version here
DATA_VERSION = "2026-08-26"

DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"

print("Using data version:", DATA_VERSION)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

In [ ]:
# List files in the raw data directory
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))

In [ ]:
import json

# Define the raw HIL-005 file
RAW_FILE = RAW_DIR / "HIL-005.json"

# Load the raw data
with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Loaded:", RAW_FILE.name)
print("Top-level type:", type(raw_data).__name__)

In [ ]:
# Inspect top-level keys and their value types
for key, value in raw_data.items():
    print(f"{key}: {type(value).__name__}")

In [ ]:
import pandas as pd

# Extract feature attributes into a DataFrame
df = pd.DataFrame(
    [feature["attributes"] for feature in raw_data["features"]]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
# Consider null values and data types for each column
df.info()

In [ ]:
# Distinguishing between missing values and empty values
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_percent"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values("missing_percent", ascending=False)

In [ ]:
df["SOURCE"].value_counts(dropna=False)

## `SOURCE` Assessment

The `SOURCE` field is fully populated across all 43,686 records and contains 29 distinct values. The observed values correspond to the coded values documented by the City of Hillsboro GIS layer.

The source values appear to encode the provenance and approximate date of the imagery or data source used to establish building information. The distribution is highly uneven, with `3DiJUL1999` accounting for approximately 47% of records.

### Cleaning Decision

No cleaning is currently required for the `SOURCE` field. The original source values will be preserved as provided by the City of Hillsboro GIS layer.

If human-readable source descriptions or temporal analysis are needed later, those should be added as derived fields rather than replacing the original values.

In [ ]:
df["YEAR_BUILT"].describe()

In [ ]:
# The mean is likely reduced from the presence of "0" values, and that the max is greater than the current year
print("YEAR_BUILT = 0:", (df["YEAR_BUILT"] == 0).sum())
print("YEAR_BUILT > 2026:", (df["YEAR_BUILT"] > 2026).sum())

In [ ]:
# What does the building with a YEAR_BUILT greater than 2026 look like? Are there any other anomalies in the data?
df.loc[df["YEAR_BUILT"] > 2026]

## `YEAR_BUILT` Anomaly Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0` and one record with a value of `2029`.

The 2029 record has:
- `STATUS = 0` (Active)
- `SOURCE = Site Plan`
- `PERMIT_ID = CMB25-00176`

The combination of `SOURCE = Site Plan` and the presence of a building permit identifier provides context for the future year, but does not establish why `YEAR_BUILT` is recorded as 2029.

### Cleaning Decision

The 2029 value will be preserved. It will be treated as an anomalous value for quality-control purposes rather than automatically classified as erroneous or replaced with a null value.

In [ ]:
# What are the sources of the buildings with a YEAR_BUILT of 0? Are there any other anomalies in the data?
# Start by looking at the sources of the buildings with a YEAR_BUILT of 0
df.loc[df["YEAR_BUILT"] == 0, "SOURCE"].value_counts()

In [ ]:
# Calculate the percentage of buildings with YEAR_BUILT = 0 for each source
zero_by_source = df["SOURCE"].where(df["YEAR_BUILT"] == 0).value_counts()
total_by_source = df["SOURCE"].value_counts()

zero_percent_by_source = (
    zero_by_source / total_by_source * 100
).sort_values(ascending=False)

zero_percent_by_source

## `YEAR_BUILT` Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0`, representing approximately 20% of the dataset. The value occurs across 25 of the 29 observed `SOURCE` values, but its prevalence varies substantially by source.

Some sources contain `YEAR_BUILT = 0` for nearly all records, while others contain very few or no zero values. This indicates that the use or availability of construction-year information varies by source.

### Cleaning Decision

No values in `YEAR_BUILT` will be modified at this stage. The original values, including `0` and `2029`, will be preserved while the meaning of the zero-value convention is investigated further.